# 5. Classificazione Supervisionata Leakage-Aware

Questa e' un'estensione metodologica, non il cuore del progetto. Il target `planet_type` e' utile ma rischia leakage perche' massa e raggio sono quasi definitori della classe.

Confrontiamo:

- **Setup A (Full)**: feature fisiche complete (planetarie/orbitali + stellari + contestuali);
- **Setup B (Ablation)**: rimuove massa/raggio planetario.

### Nota metodologica sul preprocessing supervisionato

Il file `X_scaled.csv` prodotto in NB01 serve per EDA e clustering distance-based. La parte supervisionata di questo notebook **non usa `X_scaled.csv` come input dei modelli**: legge `df_master.csv`, seleziona le feature non scalate e costruisce una `Pipeline` con:

1. `SimpleImputer(strategy="median")`;
2. `StandardScaler()`;
3. classificatore.

Questa pipeline viene passata a `GridSearchCV`, quindi imputazione e scaling sono fittati solo sul training fold a ogni iterazione della CV. Non c'e' leakage da standardizzazione globale.

### Nota su star_vmag

La feature `star_vmag` (magnitudine visuale della stella ospite) e' stata esclusa dalle feature supervisionate in NB01. Motivazione: `star_vmag` e' una misura **osservativa** che dipende fortemente dalla distanza della stella dalla Terra (correlazione con `dist_from_earth_pc_log` = 0.69). Non rappresenta una proprieta' fisica intrinseca del sistema planetario: due stelle identiche fisicamente ma a distanze diverse avranno magnitudini apparenti diverse. Includerla avrebbe introdotto un bias osservativo nel classificatore, non informazione fisica.

Le feature supervisionate usate sono quindi: 6 planetarie/orbitali + 6 stellari + 4 contestuali = **16 feature totali**.


### Ruolo della distanza

`dist_from_earth_pc_log` e' mantenuta esclusivamente come proxy esplicita degli effetti di selezione del catalogo e non viene interpretata come predittore causale del tipo planetario. `star_vmag` e' esclusa perche', oltre a non essere intrinseca, e' un proxy osservativo ridondante rispetto alla distanza (correlazione 0.69).


In [1]:
from pathlib import Path
import os

# Make the notebook robust both when executed from the project root and from
# notebook_final.
if Path.cwd().name != "notebook_final" and (Path.cwd() / "notebook_final").exists():
    os.chdir(Path.cwd() / "notebook_final")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "input" / "nasa_exoplanet_intelligence.csv"

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
processed_dir = Path("data/processed")
reports_tables = Path("reports/tables")
reports_figures = Path("reports/figures")
reports_tables.mkdir(parents=True, exist_ok=True)
reports_figures.mkdir(parents=True, exist_ok=True)

df_master = pd.read_csv(processed_dir / "df_master.csv")
with open(processed_dir / "preprocessing_metadata.json", encoding="utf-8") as f:
    prep_meta = json.load(f)

print("Distribuzione target grezzo:")
display(df_master["planet_type"].value_counts(dropna=False).to_frame("count"))


Distribuzione target grezzo:


,count
planet_type,
Mini-Neptune,2148
Gas Giant,1734
Super-Earth,1185
Neptune-like,479
Super-Jupiter,324
Sub-Earth,230
Unknown,50


## 5.1 Target e feature set supervisionati


In [2]:
EXCLUDE_TARGET_CLASSES = ["Unknown"]
valid_mask = df_master["planet_type"].notna() & ~df_master["planet_type"].isin(EXCLUDE_TARGET_CLASSES)

class_counts = df_master.loc[valid_mask, "planet_type"].value_counts()
valid_classes = class_counts[class_counts >= 30].index.tolist()
final_mask = valid_mask & df_master["planet_type"].isin(valid_classes)

SUPERVISED_FEATURES = [
    c for c in prep_meta["planetary_orbital_features"]
    + prep_meta["stellar_features"]
    + prep_meta["supervised_context_features"]
    if c in df_master.columns
]

X_all = df_master.loc[final_mask, SUPERVISED_FEATURES].reset_index(drop=True)
y_raw = df_master.loc[final_mask, "planet_type"].reset_index(drop=True)

le = LabelEncoder()
y = le.fit_transform(y_raw)
TARGET_NAMES = list(le.classes_)

LEAKAGE_KEYWORDS = ["planet_radius", "planet_mass", "planet_density", "density"]
LEAKAGE_COLS = [
    c for c in X_all.columns
    if any(keyword in c.lower() for keyword in LEAKAGE_KEYWORDS)
]

X_A = X_all.copy()
X_B = X_all.drop(columns=LEAKAGE_COLS)

print(f"Campioni supervisionati: {X_all.shape[0]}")
print(f"Classi: {TARGET_NAMES}")
print(f"Setup A feature: {X_A.shape[1]}")
print(f"Setup B feature: {X_B.shape[1]}")
print(f"Feature rimosse nel Setup B: {LEAKAGE_COLS}")
display(y_raw.value_counts().to_frame("count"))


Campioni supervisionati: 6100
Classi: ['Gas Giant', 'Mini-Neptune', 'Neptune-like', 'Sub-Earth', 'Super-Earth', 'Super-Jupiter']
Setup A feature: 16
Setup B feature: 14
Feature rimosse nel Setup B: ['planet_radius_earth_log', 'planet_mass_earth_log']


,count
planet_type,
Mini-Neptune,2148
Gas Giant,1734
Super-Earth,1185
Neptune-like,479
Super-Jupiter,324
Sub-Earth,230


## 5.2 Split stratificato


In [3]:
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

y_train = y[train_idx]
y_test = y[test_idx]

XA_train, XA_test = X_A.iloc[train_idx], X_A.iloc[test_idx]
XB_train, XB_test = X_B.iloc[train_idx], X_B.iloc[test_idx]

print(f"Train: {len(train_idx)} | Test: {len(test_idx)}")
print("Distribuzione classi train:")
display(pd.Series(y_train).map(dict(enumerate(TARGET_NAMES))).value_counts(normalize=True).round(3))


Train: 4880 | Test: 1220
Distribuzione classi train:


Mini-Neptune     0.352
Gas Giant        0.284
Super-Earth      0.194
Neptune-like     0.078
Super-Jupiter    0.053
Sub-Earth        0.038
Name: proportion, dtype: float64

## 5.3 Modelli, griglie e scoring


In [4]:
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


MODEL_GRIDS = {
    "Logistic Regression": (
        LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED),
        {"model__C": [0.1, 1.0, 10.0]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
        {"model__max_depth": [None, 5, 10, 15], "model__min_samples_leaf": [1, 5, 10]},
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1),
        {"model__n_estimators": [100, 200], "model__max_depth": [None, 10, 20], "model__min_samples_leaf": [1, 5]},
    ),
    "SVM": (
        SVC(kernel="rbf", class_weight="balanced", random_state=SEED),
        {"model__C": [0.1, 1.0, 10.0], "model__gamma": ["scale", "auto"]},
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"model__n_neighbors": [5, 7, 11], "model__weights": ["uniform", "distance"]},
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
}
print("Modelli:", list(MODEL_GRIDS.keys()))


Modelli: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'KNN']


## 5.4 GridSearchCV e test finale


In [5]:
def evaluate_setup(X_train, X_test, y_train, y_test, setup_name):
    rows = []
    best_estimators = {}

    for model_name, (model, grid) in MODEL_GRIDS.items():
        print(f"[{setup_name}] fitting {model_name}...")
        gs = GridSearchCV(
            make_pipeline(model),
            param_grid=grid,
            scoring=scoring,
            refit="f1_macro",
            cv=cv,
            n_jobs=-1,
            return_train_score=False,
        )
        gs.fit(X_train, y_train)
        y_pred = gs.predict(X_test)

        best_idx = gs.best_index_
        row = {
            "setup": setup_name,
            "model": model_name,
            "cv_accuracy_mean": gs.cv_results_["mean_test_accuracy"][best_idx],
            "cv_accuracy_std": gs.cv_results_["std_test_accuracy"][best_idx],
            "cv_f1_macro_mean": gs.cv_results_["mean_test_f1_macro"][best_idx],
            "cv_f1_macro_std": gs.cv_results_["std_test_f1_macro"][best_idx],
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_macro_f1": f1_score(y_test, y_pred, average="macro"),
            "best_params": gs.best_params_,
        }
        rows.append(row)
        best_estimators[model_name] = gs.best_estimator_
        print(f"  CV Macro-F1={row['cv_f1_macro_mean']:.3f} | Test Macro-F1={row['test_macro_f1']:.3f}")

    return pd.DataFrame(rows), best_estimators


results_A, estimators_A = evaluate_setup(XA_train, XA_test, y_train, y_test, "A_full")
results_B, estimators_B = evaluate_setup(XB_train, XB_test, y_train, y_test, "B_ablation")

results_all = pd.concat([results_A, results_B], ignore_index=True)
display(results_all.sort_values(["setup", "test_macro_f1"], ascending=[True, False]).round(4))


[A_full] fitting Logistic Regression...


  CV Macro-F1=0.923 | Test Macro-F1=0.921
[A_full] fitting Decision Tree...


  CV Macro-F1=0.999 | Test Macro-F1=1.000
[A_full] fitting Random Forest...


  CV Macro-F1=0.997 | Test Macro-F1=0.999
[A_full] fitting SVM...


  CV Macro-F1=0.878 | Test Macro-F1=0.905
[A_full] fitting KNN...


  CV Macro-F1=0.656 | Test Macro-F1=0.660
[B_ablation] fitting Logistic Regression...
  CV Macro-F1=0.426 | Test Macro-F1=0.427
[B_ablation] fitting Decision Tree...


  CV Macro-F1=0.434 | Test Macro-F1=0.431
[B_ablation] fitting Random Forest...


  CV Macro-F1=0.527 | Test Macro-F1=0.490
[B_ablation] fitting SVM...


  CV Macro-F1=0.488 | Test Macro-F1=0.448
[B_ablation] fitting KNN...
  CV Macro-F1=0.478 | Test Macro-F1=0.455


,setup,model,cv_accuracy_mean,cv_accuracy_std,cv_f1_macro_mean,cv_f1_macro_std,test_accuracy,test_macro_f1,best_params
1,A_full,Decision Tree,0.9996,0.0005,0.9992,0.0011,1.0000,1.0000,"{'model__max_depth': None, 'model__min_samples..."
2,A_full,Random Forest,0.9984,0.0005,0.9970,0.0016,0.9992,0.9989,"{'model__max_depth': 10, 'model__min_samples_l..."
0,A_full,Logistic Regression,0.9443,0.0071,0.9228,0.0080,0.9459,0.9208,{'model__C': 10.0}
3,A_full,SVM,0.9154,0.0067,0.8780,0.0057,0.9344,0.9046,"{'model__C': 10.0, 'model__gamma': 'scale'}"
4,A_full,KNN,0.7725,0.0127,0.6556,0.0187,0.7746,0.6604,"{'model__n_neighbors': 5, 'model__weights': 'd..."
7,B_ablation,Random Forest,0.6246,0.0199,0.5265,0.0201,0.5959,0.4904,"{'model__max_depth': None, 'model__min_samples..."
9,B_ablation,KNN,0.5961,0.0118,0.4779,0.0224,0.5836,0.4548,"{'model__n_neighbors': 5, 'model__weights': 'd..."
8,B_ablation,SVM,0.5582,0.0201,0.4879,0.0119,0.5172,0.4484,"{'model__C': 10.0, 'model__gamma': 'scale'}"
6,B_ablation,Decision Tree,0.5016,0.0196,0.4339,0.0248,0.4984,0.4315,"{'model__max_depth': 10, 'model__min_samples_l..."
5,B_ablation,Logistic Regression,0.4766,0.0196,0.4265,0.0155,0.4779,0.4270,{'model__C': 1.0}


## 5.5 Confronto full vs ablation


### Nota: KNN nel Setup A

Il KNN ottiene Macro-F1 di circa **0.66 nel Setup A**, molto al di sotto di Decision Tree (1.00) e Random Forest (0.98). Questo non e' sorprendente per tre motivi:

1. **Curse of dimensionality**: KNN e' un metodo instance-based che calcola distanze nello spazio delle feature. Con 17 feature, le distanze diventano meno discriminative (tutti i punti tendono ad essere equidistanti).
2. **Classi sbilanciate**: `Mini-Neptune` rappresenta ~35% del training set, `Sub-Earth` solo ~4%. KNN senza `class_weight` tende a favorire le classi maggioritarie, riducendo il recall delle classi rare.
3. **Struttura non locale**: `planet_type` dipende fortemente da massa e raggio (feature continue a range ampio). Le soglie di classificazione non sono locali nel senso di KNN.

Nel Setup B il KNN migliora relativamente (da 0.66 a 0.46), ma rimane il modello peggiore. Questo e' coerente con la sua dipendenza dalla metrica di distanza, che soffre sia dell'alta dimensionalita' sia della rimozione di feature discriminative.


In [6]:
results_A.to_csv(reports_tables / "supervised_results_setup_A.csv", index=False)
results_B.to_csv(reports_tables / "supervised_results_setup_B.csv", index=False)
results_all.to_csv(reports_tables / "supervised_results_all.csv", index=False)

comparison_ablation = results_all.pivot_table(index="model", columns="setup", values="test_macro_f1")
comparison_ablation["delta_A_minus_B"] = comparison_ablation["A_full"] - comparison_ablation["B_ablation"]
comparison_ablation = comparison_ablation.sort_values("delta_A_minus_B", ascending=False)
comparison_ablation.to_csv(reports_tables / "supervised_ablation_comparison.csv")
display(comparison_ablation.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric in zip(axes, ["test_macro_f1", "test_accuracy"]):
    sns.barplot(data=results_all, x="model", y=metric, hue="setup", ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=30)
    if ax is not axes[-1]:
        ax.get_legend().remove()
plt.suptitle("Setup A vs Setup B - test set")
plt.tight_layout()
plt.savefig(reports_figures / "supervised_setup_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


setup,A_full,B_ablation,delta_A_minus_B
model,,,
Decision Tree,1.0000,0.4315,0.5685
Random Forest,0.9989,0.4904,0.5085
Logistic Regression,0.9208,0.4270,0.4939
SVM,0.9046,0.4484,0.4562
KNN,0.6604,0.4548,0.2056


## 5.6 Confusion matrix e report del miglior Setup B


In [7]:
best_b_row = results_B.sort_values("test_macro_f1", ascending=False).iloc[0]
best_b_name = best_b_row["model"]
best_b_model = estimators_B[best_b_name]
y_pred_b = best_b_model.predict(XB_test)

print(f"Miglior modello Setup B: {best_b_name}")
print(best_b_row[["test_accuracy", "test_macro_f1"]])

report_dict = classification_report(y_test, y_pred_b, target_names=TARGET_NAMES, output_dict=True)
report_df = pd.DataFrame(report_dict).T
report_df.to_csv(reports_tables / "classification_report_best_B.csv")
display(report_df.round(3))

cm = confusion_matrix(y_test, y_pred_b)
plt.figure(figsize=(max(8, len(TARGET_NAMES) * 1.2), max(6, len(TARGET_NAMES))))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES, linewidths=0.5)
plt.title(f"Confusion matrix - {best_b_name} - Setup B")
plt.xlabel("predicted")
plt.ylabel("true")
plt.tight_layout()
plt.savefig(reports_figures / "confusion_matrix_best_B.png", dpi=300, bbox_inches="tight")
plt.show()


Miglior modello Setup B: Random Forest
test_accuracy    0.595902
test_macro_f1    0.490354
Name: 2, dtype: object


,precision,recall,f1-score,support
Gas Giant,0.794,0.732,0.762,347.000
Mini-Neptune,0.623,0.662,0.642,429.000
Neptune-like,0.213,0.167,0.187,96.000
Sub-Earth,0.297,0.239,0.265,46.000
Super-Earth,0.472,0.494,0.482,237.000
Super-Jupiter,0.536,0.692,0.604,65.000
accuracy,0.596,0.596,0.596,0.596
macro avg,0.489,0.498,0.490,1220.000
weighted avg,0.593,0.596,0.593,1220.000


## 5.7 Corrected resampled t-test e Confidence Intervals

Il confronto statistico usa strumenti trattati nel corso:

- **Corrected resampled t-test**: confronta a coppie le differenze di Macro-F1 ottenute sugli stessi split di una repeated stratified 5-fold cross-validation (2 ripetizioni). La correzione considera la dipendenza dovuta alla sovrapposizione dei training set. Le configurazioni selezionate dalla GridSearch principale vengono mantenute fisse, clonate e rifittate in ogni split: il test e' quindi un confronto condizionato alle configurazioni scelte, non una nuova stima di model selection.
- **Confidence Intervals (95%)** sulla Macro-F1 del test holdout: stimati con bootstrap per quantificare l'incertezza delle stime puntuali.


In [8]:
from scipy.stats import t as student_t

# Corrected resampled t-test sul Setup B
print("=== Corrected resampled t-test - Setup B ===")
print("H0: la differenza media di Macro-F1 tra i due modelli e' zero\n")

N_SPLITS = 5
N_REPEATS = 2
repeated_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

score_rows = []
for split_id, (cv_train_idx, cv_test_idx) in enumerate(repeated_cv.split(X_B, y), start=1):
    for model_name, estimator in estimators_B.items():
        fitted = clone(estimator)
        fitted.fit(X_B.iloc[cv_train_idx], y[cv_train_idx])
        cv_pred = fitted.predict(X_B.iloc[cv_test_idx])
        score_rows.append({
            "split_id": split_id,
            "model": model_name,
            "macro_f1": f1_score(y[cv_test_idx], cv_pred, average="macro", zero_division=0),
            "train_size": len(cv_train_idx),
            "test_size": len(cv_test_idx),
        })

repeated_scores_df = pd.DataFrame(score_rows)
repeated_scores_df.to_csv(reports_tables / "repeated_cv_macro_f1_setup_B.csv", index=False)
score_matrix = repeated_scores_df.pivot(index="split_id", columns="model", values="macro_f1")
display(score_matrix.agg(["mean", "std"]).T.sort_values("mean", ascending=False).round(4))

model_names = list(estimators_B.keys())
comparison_rows = []
n_observations = len(score_matrix)
test_train_ratio = (1.0 / N_SPLITS) / (1.0 - 1.0 / N_SPLITS)

for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        m1, m2 = model_names[i], model_names[j]
        differences = (score_matrix[m1] - score_matrix[m2]).to_numpy()
        mean_difference = float(np.mean(differences))
        variance = float(np.var(differences, ddof=1))
        corrected_se = float(np.sqrt((1.0 / n_observations + test_train_ratio) * variance))
        if corrected_se == 0.0:
            t_statistic = 0.0 if mean_difference == 0.0 else np.sign(mean_difference) * np.inf
            p_value = 1.0 if mean_difference == 0.0 else 0.0
        else:
            t_statistic = mean_difference / corrected_se
            p_value = float(2 * student_t.sf(abs(t_statistic), df=n_observations - 1))
        comparison_rows.append({
            "model_1": m1,
            "model_2": m2,
            "mean_macro_f1_difference_model1_minus_model2": mean_difference,
            "corrected_standard_error": corrected_se,
            "t_statistic": t_statistic,
            "degrees_of_freedom": n_observations - 1,
            "p_value_two_sided": p_value,
            "significant_p_lt_0_05": p_value < 0.05,
            "n_splits_total": n_observations,
            "test_train_ratio": test_train_ratio,
        })

corrected_ttest_df = pd.DataFrame(comparison_rows)
corrected_ttest_df.to_csv(reports_tables / "corrected_resampled_ttest_setup_B.csv", index=False)
display(corrected_ttest_df.round(4))

# Confidence intervals 95% sulla Macro-F1 tramite bootstrap del test set
print("\n=== Confidence Intervals 95% - Macro-F1 sul test set (bootstrap, n=1000) ===")
N_BOOT = 1000
rng = np.random.default_rng(SEED)
preds_B = {model_name: est.predict(XB_test) for model_name, est in estimators_B.items()}

ci_rows = []
all_predictions = {
    "A_full": {model_name: est.predict(XA_test) for model_name, est in estimators_A.items()},
    "B_ablation": preds_B,
}

for setup_name, preds_dict in all_predictions.items():
    for model_name, y_pred in preds_dict.items():
        boot_f1s = []
        for _ in range(N_BOOT):
            idx = rng.integers(0, len(y_test), size=len(y_test))
            boot_f1s.append(f1_score(y_test[idx], y_pred[idx], average="macro", zero_division=0))
        ci_low = float(np.percentile(boot_f1s, 2.5))
        ci_high = float(np.percentile(boot_f1s, 97.5))
        point_f1 = float(f1_score(y_test, y_pred, average="macro", zero_division=0))
        ci_rows.append({
            "setup": setup_name,
            "model": model_name,
            "macro_f1": point_f1,
            "ci_95_low": ci_low,
            "ci_95_high": ci_high,
            "ci_width": ci_high - ci_low,
        })

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(reports_tables / "confidence_intervals_macro_f1.csv", index=False)
display(ci_df.sort_values(["setup", "macro_f1"], ascending=[True, False]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, setup_name in zip(axes, ["A_full", "B_ablation"]):
    sub = ci_df[ci_df["setup"] == setup_name].sort_values("macro_f1", ascending=True).copy()
    xerr = np.vstack([
        sub["macro_f1"] - sub["ci_95_low"],
        sub["ci_95_high"] - sub["macro_f1"],
    ])
    ax.barh(sub["model"], sub["macro_f1"], xerr=xerr, color="steelblue", alpha=0.78, capsize=5)
    ax.set_xlabel("Macro-F1")
    ax.set_title(f"Macro-F1 con CI 95% - {setup_name}")
    ax.set_xlim(0, 1.05)
    ax.grid(axis="x", alpha=0.25)
plt.suptitle("Confidence intervals bootstrap (n=1000) sulla Macro-F1")
plt.tight_layout()
plt.savefig(reports_figures / "confidence_intervals_macro_f1.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nInterpretazione sintetica:")
sig = corrected_ttest_df[corrected_ttest_df["significant_p_lt_0_05"]]
print(f"- {len(sig)} confronto/i su {len(corrected_ttest_df)} risultano significativi al 5% con il corrected resampled t-test.")
print("- I test sono esplorativi e condizionati alle configurazioni selezionate dalla GridSearch principale.")
print("- I confidence intervals quantificano l'incertezza dei Macro-F1 sul test holdout.")


=== Corrected resampled t-test - Setup B ===
H0: la differenza media di Macro-F1 tra i due modelli e' zero



,mean,std
model,,
Random Forest,0.5199,0.0138
KNN,0.4831,0.0136
SVM,0.4814,0.0153
Decision Tree,0.4463,0.0214
Logistic Regression,0.4261,0.0158


,model_1,model_2,mean_macro_f1_difference_model1_minus_model2,corrected_standard_error,t_statistic,degrees_of_freedom,p_value_two_sided,significant_p_lt_0_05,n_splits_total,test_train_ratio
0,Logistic Regression,Decision Tree,-0.0202,0.0148,-1.3655,9,0.2053,False,10,0.25
1,Logistic Regression,Random Forest,-0.0938,0.0089,-10.5441,9,0.0000,True,10,0.25
2,Logistic Regression,SVM,-0.0554,0.0067,-8.2825,9,0.0000,True,10,0.25
3,Logistic Regression,KNN,-0.0570,0.0096,-5.9510,9,0.0002,True,10,0.25
4,Decision Tree,Random Forest,-0.0736,0.0105,-7.0187,9,0.0001,True,10,0.25
5,Decision Tree,SVM,-0.0351,0.0103,-3.4039,9,0.0078,True,10,0.25
6,Decision Tree,KNN,-0.0368,0.0087,-4.2293,9,0.0022,True,10,0.25
7,Random Forest,SVM,0.0385,0.0073,5.2487,9,0.0005,True,10,0.25
8,Random Forest,KNN,0.0368,0.0055,6.7536,9,0.0001,True,10,0.25
9,SVM,KNN,-0.0017,0.0073,-0.2260,9,0.8263,False,10,0.25



=== Confidence Intervals 95% - Macro-F1 sul test set (bootstrap, n=1000) ===


,setup,model,macro_f1,ci_95_low,ci_95_high,ci_width
1,A_full,Decision Tree,1.0000,1.0000,1.0000,0.0000
2,A_full,Random Forest,0.9989,0.9962,1.0000,0.0038
0,A_full,Logistic Regression,0.9208,0.9003,0.9390,0.0386
3,A_full,SVM,0.9046,0.8794,0.9257,0.0463
4,A_full,KNN,0.6604,0.6200,0.6965,0.0764
7,B_ablation,Random Forest,0.4904,0.4575,0.5227,0.0652
9,B_ablation,KNN,0.4548,0.4215,0.4912,0.0696
8,B_ablation,SVM,0.4484,0.4199,0.4790,0.0591
6,B_ablation,Decision Tree,0.4315,0.4009,0.4602,0.0593
5,B_ablation,Logistic Regression,0.4270,0.3988,0.4552,0.0564



Interpretazione sintetica:
- 8 confronto/i su 10 risultano significativi al 5% con il corrected resampled t-test.
- I test sono esplorativi e condizionati alle configurazioni selezionate dalla GridSearch principale.
- I confidence intervals quantificano l'incertezza dei Macro-F1 sul test holdout.


## 5.8 Interpretabilita' Setup B


In [9]:
feature_names_B = list(X_B.columns)

def get_inner_model(pipe):
    return pipe.named_steps["model"]

for model_name in ["Decision Tree", "Random Forest"]:
    model = get_inner_model(estimators_B[model_name])
    if hasattr(model, "feature_importances_"):
        fi = pd.DataFrame({
            "feature": feature_names_B,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
        fi.to_csv(reports_tables / f"feature_importance_B_{model_name.replace(' ', '_').lower()}.csv", index=False)
        display(fi.head(12))

        plt.figure(figsize=(10, 5))
        sns.barplot(data=fi.head(12), x="importance", y="feature", palette="magma")
        plt.title(f"Feature importance - {model_name} - Setup B")
        plt.tight_layout()
        plt.savefig(reports_figures / f"feature_importance_B_{model_name.replace(' ', '_').lower()}.png", dpi=300, bbox_inches="tight")
        plt.show()

lr_model = get_inner_model(estimators_B["Logistic Regression"])
coef_df = pd.DataFrame({
    "feature": feature_names_B,
    "abs_mean_coefficient": np.mean(np.abs(lr_model.coef_), axis=0),
}).sort_values("abs_mean_coefficient", ascending=False)
coef_df.to_csv(reports_tables / "logreg_coefficients_B.csv", index=False)
display(coef_df.head(12))

plt.figure(figsize=(10, 5))
sns.barplot(data=coef_df.head(12), x="abs_mean_coefficient", y="feature", palette="mako")
plt.title("Coefficienti medi assoluti - Logistic Regression - Setup B")
plt.tight_layout()
plt.savefig(reports_figures / "logreg_coefficients_B.png", dpi=300, bbox_inches="tight")
plt.show()


,feature,importance
0,equilibrium_temp_k,0.211121
3,semi_major_axis_au_log,0.161740
2,orbital_period_days_log,0.128211
13,dist_from_earth_pc_log,0.096829
5,star_radius_sun,0.074231
1,orbital_eccentricity,0.069739
9,star_metallicity,0.053125
7,star_age_gyr,0.050912
8,star_surface_gravity,0.049228
11,n_planets,0.026392


,feature,importance
0,equilibrium_temp_k,0.141626
2,orbital_period_days_log,0.138399
3,semi_major_axis_au_log,0.129705
13,dist_from_earth_pc_log,0.092663
5,star_radius_sun,0.080035
6,star_mass_sun,0.072553
8,star_surface_gravity,0.071423
9,star_metallicity,0.061695
4,star_temp_k,0.061528
7,star_age_gyr,0.051036


,feature,abs_mean_coefficient
3,semi_major_axis_au_log,1.232202
5,star_radius_sun,1.215903
2,orbital_period_days_log,0.695742
0,equilibrium_temp_k,0.540816
8,star_surface_gravity,0.441256
1,orbital_eccentricity,0.381704
11,n_planets,0.310020
4,star_temp_k,0.297325
9,star_metallicity,0.288214
12,multi_planet_system,0.245823


## 5.9 Salvataggio riepilogo supervised


In [10]:
summary = {
    "target": "planet_type",
    "excluded_target_classes": EXCLUDE_TARGET_CLASSES,
    "target_names": TARGET_NAMES,
    "n_samples": int(len(y)),
    "setup_A_features": list(X_A.columns),
    "setup_B_features": list(X_B.columns),
    "ablation_removed_features": LEAKAGE_COLS,
    "best_A_by_test_macro_f1": results_A.sort_values("test_macro_f1", ascending=False).iloc[0].to_dict(),
    "best_B_by_test_macro_f1": results_B.sort_values("test_macro_f1", ascending=False).iloc[0].to_dict(),
    "additional_statistical_outputs": [
        "repeated_cv_macro_f1_setup_B.csv",
        "corrected_resampled_ttest_setup_B.csv",
        "confidence_intervals_macro_f1.csv",
        "confidence_intervals_macro_f1.png",
    ],
}

def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

with open(reports_tables / "supervised_summary.json", "w", encoding="utf-8") as f:
    json.dump(json_safe(summary), f, indent=4, ensure_ascii=False)

print("Risultati supervised salvati in reports/tables e reports/figures")


Risultati supervised salvati in reports/tables e reports/figures
